# Mercado de Limones Argentino — Análisis Causal
## Notebook 03: ISA ~ Volumen 0km

**Período:** Enero 2023 — Mayo 2026 (41 meses)  
**Autor:** Rodolfo Gabriel Riveros Lobos  
**Fecha:** Junio 2026

---

### Objetivo
Testear formalmente si el volumen de patentamientos 0km **precede temporalmente** 
al Índice de Selección Adversa (ISA) del mercado de usados, consistente con la 
hipótesis del efecto cascada de Akerlof (1970).

### Hoja de ruta
1. Carga de series procesadas
2. Exploración visual
3. Test de estacionariedad (ADF)
4. Test de cointegración (Engle-Granger)
5. Función de Correlación Cruzada (CCF)
6. Test de Causalidad de Granger
7. Modelo ADL — cuantificación del efecto
8. Análisis de sensibilidad del ISA
9. Hallazgos y limitaciones

In [1]:
# ============================================================
# CELDA 1 — Setup
# ============================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
from config import PATHS
from metricas import calcular_isa
from visualizacion import (
    graficar_series_temporales,
    graficar_ccf,
    graficar_sensibilidad_isa,
    graficar_residuos_adl,
)

FIGURES_PATH = PROJECT_ROOT / "reports" / "figures"

print("✅ Módulos cargados")
print(f"   Figures → {FIGURES_PATH}")

✅ Módulos cargados
   Figures → c:\Users\usuario\Documents\PORTFOLIO\mercado-limones-causal\reports\figures


In [3]:
# ============================================================
# CELDA 2 — Carga de series
# ============================================================

serie_isa = pd.read_parquet(PATHS["processed"] / "serie_isa.parquet")
serie_0km = pd.read_parquet(PATHS["processed"] / "serie_0km.parquet")

serie_isa["fecha"] = pd.to_datetime(serie_isa["periodo"])
serie_0km["fecha"] = pd.to_datetime(serie_0km["periodo"])

assert len(serie_isa) == 41, f"Esperado 41 filas, got {len(serie_isa)}"
assert len(serie_0km) == 41, f"Esperado 41 filas, got {len(serie_0km)}"
assert serie_isa["periodo"].iloc[0] == serie_0km["periodo"].iloc[0], \
    "Las series no comienzan en el mismo período"

print("✅ Series cargadas")
print(f"   ISA → {serie_isa['periodo'].iloc[0]} : {serie_isa['periodo'].iloc[-1]}")
print(f"   0km → {serie_0km['periodo'].iloc[0]} : {serie_0km['periodo'].iloc[-1]}")

✅ Series cargadas
   ISA → 2023-01 : 2026-05
   0km → 2023-01 : 2026-05


## 2. Exploración visual

Antes de cualquier test estadístico, la inspección visual es obligatoria.  
Un modelo que contradice lo observable tiene un error — no la vista.

**Preguntas guía:**
- ¿Las series se mueven juntas o de forma independiente?
- ¿Hay un desfase visible entre el 0km y el ISA?
- ¿Se identifica el shock de enero 2024 en ambas series?

In [4]:
# ============================================================
# CELDA 3 — Exploración visual
# ============================================================
# Propósito: antes de cualquier test estadístico, la inspección
# visual es obligatoria. Un modelo que contradice lo visible
# tiene un error — no la vista.
# ============================================================

graficar_series_temporales(serie_isa, serie_0km, FIGURES_PATH)

✅ Figura guardada: c:\Users\usuario\Documents\PORTFOLIO\mercado-limones-causal\reports\figures\01_series_temporales.png


## 3. Estacionariedad — condición previa al modelado

Las series económicas suelen tener tendencia propia (raíz unitaria).  
Regresar dos series no estacionarias produce **regresión espuria**: 
correlaciones altas y significativas que no reflejan ninguna relación real.

El Test de Dickey-Fuller Aumentado (ADF) verifica si cada serie tiene raíz unitaria.  
- **H₀:** la serie tiene raíz unitaria (no estacionaria)  
- Si p-value < 0.05 → rechazamos H₀ → serie estacionaria ✅  
- Si p-value > 0.05 → no rechazamos H₀ → serie no estacionaria ⚠️

In [5]:
# ============================================================
# CELDA 4 — Test de Estacionariedad (ADF)
# ============================================================
# Hipótesis nula (H0): la serie TIENE raíz unitaria (no estacionaria)
# Si p-value < 0.05 → rechazamos H0 → serie estacionaria
# Si p-value > 0.05 → no rechazamos H0 → serie no estacionaria
#                     → hay que diferenciar antes de Granger
# ============================================================

from statsmodels.tsa.stattools import adfuller

def run_adf(serie: pd.Series, nombre: str) -> dict:
    resultado = adfuller(serie.dropna(), autolag="AIC")
    print(f"\n=== ADF — {nombre} ===")
    print(f"  Estadístico ADF : {resultado[0]:.4f}")
    print(f"  p-value         : {resultado[1]:.4f}")
    print(f"  Lags usados     : {resultado[2]}")
    print(f"  Valores críticos:")
    for nivel, valor in resultado[4].items():
        print(f"    {nivel}: {valor:.4f}")
    conclusion = "✅ ESTACIONARIA" if resultado[1] < 0.05 else "⚠️  NO ESTACIONARIA"
    print(f"  Conclusión      : {conclusion}")
    return {
        "nombre":     nombre,
        "estadistico": resultado[0],
        "pvalue":      resultado[1],
        "estacionaria": resultado[1] < 0.05,
    }

resultados_adf = []
resultados_adf.append(run_adf(serie_isa["ISA"],    "ISA (nivel)"))
resultados_adf.append(run_adf(serie_0km["total"],  "Volumen 0km (nivel)"))


=== ADF — ISA (nivel) ===
  Estadístico ADF : -1.9846
  p-value         : 0.2934
  Lags usados     : 0
  Valores críticos:
    1%: -3.6056
    5%: -2.9371
    10%: -2.6070
  Conclusión      : ⚠️  NO ESTACIONARIA

=== ADF — Volumen 0km (nivel) ===
  Estadístico ADF : -1.7417
  p-value         : 0.4098
  Lags usados     : 2
  Valores críticos:
    1%: -3.6155
    5%: -2.9413
    10%: -2.6092
  Conclusión      : ⚠️  NO ESTACIONARIA


## 4. Cointegración — ¿existe una relación de largo plazo?

Ambas series son **I(1)** — no estacionarias en niveles.  
Antes de diferenciar, verificamos si están **cointegradas**: 
si existe una combinación lineal de ellas que sí sea estacionaria.

**Por qué importa:** si las series están cointegradas, diferenciarlas 
destruiría la relación de largo plazo que buscamos medir.  
La cointegración confirma que el vínculo entre mercados no es espurio.

**Procedimiento Engle-Granger (1987):**
1. Regresar ISA ~ Volumen 0km en niveles (OLS)
2. Aplicar ADF a los residuos (sin constante, por propiedad del OLS)
3. Si residuos son estacionarios → series cointegradas

In [10]:
# ============================================================
# CELDA 5 — Test de Cointegración (Engle-Granger)
# ============================================================
# Ambas series son I(1) — no estacionarias en niveles.
# Antes de diferenciar, verificamos si existe una relación
# de largo plazo estable entre ellas.
#
# Procedimiento:
#   1. Regresión OLS: ISA ~ Volumen_0km (en niveles)
#   2. ADF sobre los residuos de esa regresión
#
# Hipótesis nula (H0): residuos tienen raíz unitaria
#                      → NO hay cointegración
# Si p-value < 0.05 → rechazamos H0 → series cointegradas
# ============================================================

import numpy as np
from statsmodels.tsa.stattools import adfuller
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

# --- Paso 1: Regresión en niveles ---
y = serie_isa["ISA"].values
x = serie_0km["total"].values

x_const = add_constant(x)
modelo  = OLS(y, x_const).fit()
residuos = modelo.resid

print("=== REGRESIÓN EN NIVELES: ISA ~ Volumen_0km ===")
print(f"  Intercepto : {modelo.params[0]:.4f}")
print(f"  Coeficiente: {modelo.params[1]:.6f}")
print(f"  R²         : {modelo.rsquared:.4f}")
print(f"  p-value β₁ : {modelo.pvalues[1]:.4f}")

# --- Paso 2: ADF sobre residuos ---
print("\n=== ADF SOBRE RESIDUOS (Test de Engle-Granger) ===")
adf_resid = adfuller(residuos, autolag="AIC", regression="n")

print(f"  Estadístico ADF : {adf_resid[0]:.4f}")
print(f"  p-value         : {adf_resid[1]:.4f}")
print(f"  Lags usados     : {adf_resid[2]}")
print(f"  Valores críticos:")
for nivel, valor in adf_resid[4].items():
    print(f"    {nivel}: {valor:.4f}")

if adf_resid[1] < 0.05:
    print("\n✅ COINTEGRACIÓN DETECTADA")
    print("   Los residuos son estacionarios.")
    print("   Existe una relación de largo plazo entre 0km e ISA.")
    print("   → Trabajar en niveles es válido.")
    print("   → Considerar modelo ECM además del ADL.")
else:
    print("\n⚠️  SIN COINTEGRACIÓN")
    print("   Los residuos tienen raíz unitaria.")
    print("   → Diferenciar ambas series antes del Test de Granger.")
    print("   → Se pierde información de largo plazo (documentar limitación).")

=== REGRESIÓN EN NIVELES: ISA ~ Volumen_0km ===
  Intercepto : 1.0591
  Coeficiente: -0.000030
  R²         : 0.0823
  p-value β₁ : 0.0690

=== ADF SOBRE RESIDUOS (Test de Engle-Granger) ===
  Estadístico ADF : -3.0174
  p-value         : 0.0025
  Lags usados     : 0
  Valores críticos:
    1%: -2.6239
    5%: -1.9493
    10%: -1.6115

✅ COINTEGRACIÓN DETECTADA
   Los residuos son estacionarios.
   Existe una relación de largo plazo entre 0km e ISA.
   → Trabajar en niveles es válido.
   → Considerar modelo ECM además del ADL.


## 5. Función de Correlación Cruzada (CCF)

La cointegración confirma que existe una relación estructural.  
La CCF identifica **en qué lag exacto** el volumen 0km tiene mayor 
correlación con el ISA futuro.

Este lag determina la especificación del modelo ADL en la siguiente sección.

**Interpretación:**
- Lag 0 → correlación contemporánea
- Lag k → el 0km de hace k meses predice el ISA de hoy
- Banda de significancia: ±1.96/√n al 95%

In [9]:
# ============================================================
# CELDA 6 — Función de Correlación Cruzada (CCF)
# ============================================================
# Propósito: identificar en qué rezago exacto el volumen 0km
# tiene mayor correlación con el ISA futuro.
# Esto determina los lags a incluir en el modelo ADL.
#
# Interpretación:
#   lag=0 → correlación contemporánea (ya sabemos que es débil)
#   lag=1 → ¿el 0km de este mes predice el ISA del mes siguiente?
#   lag=k → ¿el 0km de hace k meses predice el ISA de hoy?
# ============================================================

from visualizacion import graficar_ccf

graficar_ccf(serie_isa, serie_0km, FIGURES_PATH, max_lags=12)

# --- Output numérico para documentar ---
import numpy as np

isa_z = (serie_isa["ISA"].values - serie_isa["ISA"].mean()) \
        / serie_isa["ISA"].std()
vol_z = (serie_0km["total"].values - serie_0km["total"].mean()) \
        / serie_0km["total"].std()

print("\n=== CORRELACIONES CRUZADAS POR LAG ===")
print(f"{'Lag':>4}  {'Correlación':>12}  {'Significativa':>14}")
print("-" * 35)

n     = len(isa_z)
banda = 1.96 / np.sqrt(n)

for lag in range(13):
    if lag == 0:
        corr = np.corrcoef(vol_z, isa_z)[0, 1]
    else:
        corr = np.corrcoef(vol_z[:-lag], isa_z[lag:])[0, 1]
    sig = "✅ SÍ" if abs(corr) > banda else "—"
    print(f"{lag:>4}  {corr:>12.4f}  {sig:>14}")

print(f"\nBanda de significancia: ±{banda:.4f} (95%, n={n})")

✅ Figura guardada: c:\Users\usuario\Documents\PORTFOLIO\mercado-limones-causal\reports\figures\02_ccf.png

=== CORRELACIONES CRUZADAS POR LAG ===
 Lag   Correlación   Significativa
-----------------------------------
   0       -0.2869               —
   1       -0.4196            ✅ SÍ
   2       -0.4520            ✅ SÍ
   3       -0.4357            ✅ SÍ
   4       -0.3344            ✅ SÍ
   5       -0.2582               —
   6       -0.1628               —
   7       -0.1768               —
   8       -0.0307               —
   9       -0.0153               —
  10        0.0533               —
  11       -0.0939               —
  12        0.1218               —

Banda de significancia: ±0.3061 (95%, n=41)


## 6. Test de Causalidad de Granger

La CCF muestra correlación. El Test de Granger pregunta algo más fuerte:  
¿el historial del 0km **mejora la predicción** del ISA más allá de lo que 
el propio historial del ISA ya predice?

- **H₀:** el volumen 0km NO causa (en sentido Granger) al ISA
- Si p-value < 0.05 → el 0km contiene información predictiva sobre el ISA

> ⚠️ **Advertencia metodológica:** Granger ≠ causalidad estructural.  
> Mide precedencia temporal. La interpretación económica del mecanismo 
> es responsabilidad del analista.

In [11]:
# ============================================================
# CELDA 7 — Test de Causalidad de Granger
# ============================================================
# Pregunta formal: ¿el historial de 0km ayuda a predecir el ISA
# más allá de lo que el propio historial del ISA ya predice?
#
# H0: el 0km NO causa en sentido Granger al ISA
# Si p-value < 0.05 → rechazamos H0 → el 0km SÍ precede al ISA
#
# ADVERTENCIA: Granger ≠ causalidad estructural.
# Mide precedencia temporal, no mecanismo causal.
# La interpretación económica es nuestra responsabilidad.
# ============================================================

from statsmodels.tsa.stattools import grangercausalitytests
import numpy as np

# Construir matriz [ISA, 0km] — orden importa:
# la primera columna es la variable a predecir (ISA)
datos_granger = np.column_stack([
    serie_isa["ISA"].values,
    serie_0km["total"].values,
])

print("=== TEST DE CAUSALIDAD DE GRANGER ===")
print("H0: el volumen 0km NO causa (en sentido Granger) al ISA\n")

resultados = grangercausalitytests(datos_granger, maxlag=4, verbose=False)

print(f"{'Lag':>4}  {'F-stat':>8}  {'p-value':>8}  {'Conclusión'}")
print("-" * 50)

for lag, resultado in resultados.items():
    f_stat = resultado[0]["ssr_ftest"][0]
    p_val  = resultado[0]["ssr_ftest"][1]
    conclusion = "✅ Rechaza H0" if p_val < 0.05 else "— No rechaza H0"
    print(f"{lag:>4}  {f_stat:>8.4f}  {p_val:>8.4f}  {conclusion}")

print("\n⚠️  Recordatorio: Granger mide precedencia temporal, no causalidad estructural.")
print("   La interpretación económica del mecanismo es responsabilidad del analista.")

=== TEST DE CAUSALIDAD DE GRANGER ===
H0: el volumen 0km NO causa (en sentido Granger) al ISA

 Lag    F-stat   p-value  Conclusión
--------------------------------------------------
   1    4.6597    0.0374  ✅ Rechaza H0
   2    3.5153    0.0410  ✅ Rechaza H0
   3    3.6155    0.0239  ✅ Rechaza H0
   4    2.9776    0.0362  ✅ Rechaza H0

⚠️  Recordatorio: Granger mide precedencia temporal, no causalidad estructural.
   La interpretación económica del mecanismo es responsabilidad del analista.


c:\Users\usuario\.conda\envs\limones-causal\Lib\site-packages\statsmodels\tsa\stattools.py:1545: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


## 7. Modelo ADL — cuantificación del efecto

El Test de Granger confirma precedencia. El modelo ADL cuantifica:
- **Magnitud** del efecto en cada lag
- **Persistencia** del shock (componente autorregresivo)
- **Control** de la ruptura estructural 2024 (dummy de régimen)

**Especificación:**

$$ISA_t = \alpha + \beta_1 ISA_{t-1} + \gamma_1 Vol_{t-1} + \gamma_2 Vol_{t-2} + \gamma_3 Vol_{t-3} + \delta \cdot D_{2024} + \varepsilon_t$$

El diagnóstico de residuos verifica que los supuestos del OLS se cumplan.  
Residuos con estructura indican modelo mal especificado.

In [12]:
# ============================================================
# CELDA 8 — Modelo ADL (Autorregresivo de Rezagos Distribuidos)
# ============================================================
# Propósito: cuantificar el impacto y el timing exacto del
# efecto cascada. El CCF identificó el lag — el ADL lo mide.
#
# Especificación:
#   ISA(t) = α
#            + β₁·ISA(t-1)          ← componente autorregresivo
#            + γ₁·0km(t-1)          ← efecto rezagado lag 1
#            + γ₂·0km(t-2)          ← efecto rezagado lag 2 (pico CCF)
#            + γ₃·0km(t-3)          ← efecto rezagado lag 3
#            + δ·dummy_regimen       ← control ruptura estructural 2024
#            + ε(t)
#
# La dummy_regimen captura el cambio de régimen de importaciones
# para no confundirlo con el efecto cascada.
# ============================================================

import pandas as pd
import numpy as np
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

# --- Construcción del dataset ADL ---
df = pd.DataFrame({
    "ISA"    : serie_isa["ISA"].values,
    "vol_0km": serie_0km["total"].values,
    "periodo": serie_isa["periodo"].values,
}).assign(
    ISA_lag1    = lambda x: x["ISA"].shift(1),
    vol_lag1    = lambda x: x["vol_0km"].shift(1),
    vol_lag2    = lambda x: x["vol_0km"].shift(2),
    vol_lag3    = lambda x: x["vol_0km"].shift(3),
    dummy_reg   = lambda x: (x["periodo"] >= "2024-01").astype(int),
).dropna()

print(f"Observaciones disponibles tras lags: {len(df)} (de 41 originales)")

# --- Estimación OLS ---
X = add_constant(df[[
    "ISA_lag1", "vol_lag1", "vol_lag2", "vol_lag3", "dummy_reg"
]])
y = df["ISA"]

modelo_adl = OLS(y, X).fit()

print("\n=== MODELO ADL — RESULTADOS ===")
print(f"{'Variable':<15} {'Coef':>10} {'p-value':>10}  {'Sig'}")
print("-" * 45)

nombres = ["Intercepto", "ISA(t-1)", "0km(t-1)", "0km(t-2)", "0km(t-3)", "Dummy 2024"]
for nombre, coef, pval in zip(nombres,
                               modelo_adl.params,
                               modelo_adl.pvalues):
    sig = "✅" if pval < 0.05 else ("·" if pval < 0.10 else "—")
    print(f"{nombre:<15} {coef:>10.6f} {pval:>10.4f}  {sig}")

print(f"\nR²            : {modelo_adl.rsquared:.4f}")
print(f"R² ajustado   : {modelo_adl.rsquared_adj:.4f}")
print(f"AIC           : {modelo_adl.aic:.2f}")
print(f"Observaciones : {int(modelo_adl.nobs)}")

# --- Guardar residuos para diagnóstico ---
residuos_adl = pd.Series(modelo_adl.resid, name="residuos")

Observaciones disponibles tras lags: 38 (de 41 originales)

=== MODELO ADL — RESULTADOS ===
Variable              Coef    p-value  Sig
---------------------------------------------
Intercepto        1.352105     0.0491  ✅
ISA(t-1)          0.717881     0.0000  ✅
0km(t-1)         -0.000021     0.0630  ·
0km(t-2)         -0.000007     0.5317  —
0km(t-3)         -0.000002     0.8904  —
Dummy 2024       -0.206491     0.4609  —

R²            : 0.7228
R² ajustado   : 0.6795
AIC           : 87.22
Observaciones : 38


In [13]:
# ============================================================
# CELDA 9 — Diagnóstico de residuos del modelo ADL
# ============================================================
# Un modelo cuyos residuos tienen estructura (autocorrelación,
# heterocedasticidad, no normalidad) tiene especificación
# incorrecta — los coeficientes y p-values no son confiables.
# ============================================================

from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan
from scipy import stats

graficar_residuos_adl(residuos_adl, FIGURES_PATH)

# --- Tests formales ---
dw      = durbin_watson(residuos_adl)
_, pval_bp, _, _ = het_breuschpagan(residuos_adl, modelo_adl.model.exog)
_, pval_norm = stats.shapiro(residuos_adl)

print("=== DIAGNÓSTICO FORMAL DE RESIDUOS ===\n")
print(f"Durbin-Watson        : {dw:.4f}")
print(f"  Esperado ≈ 2.0 (sin autocorrelación)")
print(f"  Conclusión: {'✅ Sin autocorrelación' if 1.5 < dw < 2.5 else '⚠️  Posible autocorrelación'}\n")

print(f"Breusch-Pagan (heterocedasticidad)")
print(f"  p-value: {pval_bp:.4f}")
print(f"  Conclusión: {'✅ Homocedasticidad' if pval_bp > 0.05 else '⚠️  Heterocedasticidad detectada'}\n")

print(f"Shapiro-Wilk (normalidad)")
print(f"  p-value: {pval_norm:.4f}")
print(f"  Conclusión: {'✅ Residuos normales' if pval_norm > 0.05 else '⚠️  No normalidad — revisar outliers'}")

✅ Figura guardada: c:\Users\usuario\Documents\PORTFOLIO\mercado-limones-causal\reports\figures\04_residuos_adl.png
=== DIAGNÓSTICO FORMAL DE RESIDUOS ===

Durbin-Watson        : 1.7930
  Esperado ≈ 2.0 (sin autocorrelación)
  Conclusión: ✅ Sin autocorrelación

Breusch-Pagan (heterocedasticidad)
  p-value: 0.1597
  Conclusión: ✅ Homocedasticidad

Shapiro-Wilk (normalidad)
  p-value: 0.9340
  Conclusión: ✅ Residuos normales


### 7b. Modelo ADL Restringido

El modelo completo presenta **multicolinealidad entre rezagos**:  
los lags 1, 2 y 3 del 0km están correlacionados entre sí, inflando los 
errores estándar e impidiendo aislar el efecto de cada lag individualmente.

Especificación parsimoniosa: un solo lag (lag 2, pico del CCF).  
Se comparan ambos modelos con R² ajustado y AIC.

In [14]:
# ============================================================
# CELDA 10 — Modelo ADL Restringido (lag óptimo)
# ============================================================
# Motivación: el modelo completo sufre multicolinealidad entre
# rezagos. El CCF identificó lag 2 como óptimo (-0.4520).
# Especificación parsimoniosa: un solo lag de 0km.
#
# ISA(t) = α + β₁·ISA(t-1) + γ₂·0km(t-2) + δ·dummy_reg + ε(t)
# ============================================================

df_r = pd.DataFrame({
    "ISA"    : serie_isa["ISA"].values,
    "vol_0km": serie_0km["total"].values,
    "periodo": serie_isa["periodo"].values,
}).assign(
    ISA_lag1  = lambda x: x["ISA"].shift(1),
    vol_lag2  = lambda x: x["vol_0km"].shift(2),
    dummy_reg = lambda x: (x["periodo"] >= "2024-01").astype(int),
).dropna()

X_r = add_constant(df_r[["ISA_lag1", "vol_lag2", "dummy_reg"]])
y_r = df_r["ISA"]

modelo_r = OLS(y_r, X_r).fit()

print("=== MODELO ADL RESTRINGIDO ===")
print(f"{'Variable':<15} {'Coef':>10} {'p-value':>10}  {'Sig'}")
print("-" * 45)

nombres_r = ["Intercepto", "ISA(t-1)", "0km(t-2)", "Dummy 2024"]
for nombre, coef, pval in zip(nombres_r,
                               modelo_r.params,
                               modelo_r.pvalues):
    sig = "✅" if pval < 0.05 else ("·" if pval < 0.10 else "—")
    print(f"{nombre:<15} {coef:>10.6f} {pval:>10.4f}  {sig}")

print(f"\nR²            : {modelo_r.rsquared:.4f}")
print(f"R² ajustado   : {modelo_r.rsquared_adj:.4f}")
print(f"AIC           : {modelo_r.aic:.2f}")
print(f"Observaciones : {int(modelo_r.nobs)}")

# Comparación directa con modelo completo
print("\n=== COMPARACIÓN DE MODELOS ===")
print(f"{'Métrica':<15} {'Completo':>10} {'Restringido':>12}")
print("-" * 40)
print(f"{'R² ajustado':<15} {0.6795:>10.4f} {modelo_r.rsquared_adj:>12.4f}")
print(f"{'AIC':<15} {87.22:>10.2f} {modelo_r.aic:>12.2f}")
print(f"{'Variables':<15} {'6':>10} {str(len(modelo_r.params)):>12}")

=== MODELO ADL RESTRINGIDO ===
Variable              Coef    p-value  Sig
---------------------------------------------
Intercepto        0.644836     0.1948  —
ISA(t-1)          0.757645     0.0000  ✅
0km(t-2)         -0.000013     0.2422  —
Dummy 2024       -0.193506     0.4776  —

R²            : 0.6815
R² ajustado   : 0.6542
AIC           : 89.89
Observaciones : 39

=== COMPARACIÓN DE MODELOS ===
Métrica           Completo  Restringido
----------------------------------------
R² ajustado         0.6795       0.6542
AIC                  87.22        89.89
Variables                6            4


## 8. Análisis de sensibilidad del ISA

**Pregunta:** ¿los resultados cambian si usamos distintas ponderaciones del ISA?

Si ISA 50/50, ISA 70/30 e ISA 30/70 producen la misma conclusión → 
el hallazgo es robusto y no es un artefacto de la ponderación elegida.

Si la conclusión depende de la ponderación → **Decisión Metodológica C**: 
reportar como evidencia parcial con limitaciones explícitas.

In [15]:
# ============================================================
# CELDA 11 — Análisis de Sensibilidad
# ============================================================
# Pregunta: ¿los resultados del Test de Granger cambian si
# usamos ISA_70_30 o ISA_30_70 en lugar del ISA base?
# Si la conclusión es estable → el hallazgo no es un artefacto
# de la ponderación elegida.
# ============================================================

variantes = {
    "ISA 50/50 (base)": serie_isa["ISA"].values,
    "ISA 70/30":        serie_isa["ISA_70_30"].values,
    "ISA 30/70":        serie_isa["ISA_30_70"].values,
}

print("=== ANÁLISIS DE SENSIBILIDAD — TEST DE GRANGER (lag=2) ===")
print(f"{'Variante ISA':<20} {'F-stat':>8} {'p-value':>8}  {'Conclusión'}")
print("-" * 58)

for nombre, isa_vals in variantes.items():
    datos = np.column_stack([isa_vals, serie_0km["total"].values])
    res   = grangercausalitytests(datos, maxlag=2, verbose=False)
    f     = res[2][0]["ssr_ftest"][0]
    p     = res[2][0]["ssr_ftest"][1]
    sig   = "✅ Rechaza H0" if p < 0.05 else "— No rechaza H0"
    print(f"{nombre:<20} {f:>8.4f} {p:>8.4f}  {sig}")

print("\n⚠️  Si las tres variantes rechazan H0 → hallazgo robusto a la ponderación.")

# --- Visualización de sensibilidad ---
graficar_sensibilidad_isa(serie_isa, FIGURES_PATH)

=== ANÁLISIS DE SENSIBILIDAD — TEST DE GRANGER (lag=2) ===
Variante ISA           F-stat  p-value  Conclusión
----------------------------------------------------------
ISA 50/50 (base)       3.5153   0.0410  ✅ Rechaza H0
ISA 70/30              3.2243   0.0522  — No rechaza H0
ISA 30/70              2.4224   0.1039  — No rechaza H0

⚠️  Si las tres variantes rechazan H0 → hallazgo robusto a la ponderación.


c:\Users\usuario\.conda\envs\limones-causal\Lib\site-packages\statsmodels\tsa\stattools.py:1545: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


✅ Figura guardada: c:\Users\usuario\Documents\PORTFOLIO\mercado-limones-causal\reports\figures\03_sensibilidad_isa.png


## 9. Síntesis de hallazgos y limitaciones

> **Decisión Metodológica C — Evidencia Parcial**  
> Se detecta evidencia de precedencia temporal del volumen 0km sobre el ISA,  
> sensible a la especificación del índice y potencialmente contaminada por  
> variables macroeconómicas omitidas. El hallazgo es consistente con la  
> hipótesis del efecto cascada pero no permite establecer causalidad  
> estructural sin controles adicionales.

In [16]:
# ============================================================
# CELDA 12 — Hallazgos, Limitaciones y Próximos Pasos
# ============================================================

hallazgos = """
╔══════════════════════════════════════════════════════════════╗
║         MERCADO DE LIMONES ARGENTINO — ANÁLISIS CAUSAL      ║
║              Notebook 03 — Síntesis de Hallazgos            ║
╚══════════════════════════════════════════════════════════════╝

PERÍODO ANALIZADO: Enero 2023 — Mayo 2026 (41 meses)
RÉGIMEN BASE ISA:  2023_base (pre-shock Milei)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
HALLAZGOS CONFIRMADOS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[H1] COINTEGRACIÓN DETECTADA
     Las series ISA y Volumen 0km son individualmente no
     estacionarias I(1) pero comparten una relación estructural
     de largo plazo (Test Engle-Granger, p=0.0025).
     → Trabajar en niveles es válido. No hay regresión espuria.
     → Existe un "vínculo invisible" entre ambos mercados que
       los impide alejarse indefinidamente.

[H2] PRECEDENCIA TEMPORAL CONFIRMADA (Granger)
     El historial del volumen 0km contiene información útil
     para predecir el ISA más allá de su propio historial.
     Lags 1-4 rechazan H0 (p < 0.05).
     → El mercado primario antecede al secundario.
     → El 0km funciona como indicador adelantado del ISA.

[H3] LAG ÓPTIMO: 2 MESES
     La CCF identifica el lag 2 como el de mayor correlación
     cruzada (r=-0.452, significativa al 95%).
     Ventana de transmisión: 1 a 4 meses.
     → El efecto cascada tarda ~60 días en transmitirse
       del mercado primario al secundario.

[H4] INERCIA DEL MERCADO DE USADOS
     El componente autorregresivo ISA(t-1) domina el modelo
     ADL (coef=0.72, p<0.001, R²=0.72).
     → El mercado de usados tiene memoria fuerte.
     → Un shock tarda varios meses en disiparse.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EVIDENCIA PARCIAL (Decisión Metodológica C)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[E1] SENSIBILIDAD A LA PONDERACIÓN DEL ISA
     El resultado de Granger es significativo con ISA 50/50
     (p=0.041) pero no se sostiene con ISA 70/30 (p=0.052)
     ni ISA 30/70 (p=0.104).
     → El hallazgo no es robusto a distintas ponderaciones.
     → La señal opera principalmente vía antigüedad del parque,
       no vía composición Mercosur (pct_protocolo21).

[E2] ENDOGENEIDAD NO RESUELTA
     Tipo de cambio, inflación (IPC) y tasas de interés (Badlar)
     afectan simultáneamente a ambos mercados y no fueron
     incluidos como controles.
     → Los coeficientes del 0km pueden estar contaminados por
       variables macroeconómicas omitidas.
     → No es posible aislar el efecto cascada puro sin estos
       controles en el contexto argentino 2023-2026.

[E3] RUPTURA ESTRUCTURAL 2024
     La desregulación de importaciones (Resolución 271/2025,
     Decreto 49/2025) cambió la naturaleza de pct_protocolo21
     en el mercado de usados. El ISA calculado sobre parámetros
     2023 pierde poder interpretativo en el régimen post-2024.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LIMITACIONES ESTRUCTURALES (heredadas del proyecto antecedente)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[L1] PUNTO CIEGO DE AKERLOF
     La DNRPA registra transacciones completadas, no intentos
     fallidos. El mercado de limones que NO se transacciona
     es invisible para el modelo.

[L2] AUSENCIA DE PRECIOS
     El análisis es estrictamente volumétrico y composicional.
     No es posible cuantificar el impacto en pesos por unidad.

[L3] SUBREGISTRO ESTACIONAL
     Diciembre presenta anomalías sistemáticas por liquidación
     de stock de fin de año. Las dummies estacionales no fueron
     incluidas en esta versión del modelo.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
PRÓXIMOS PASOS (Extensiones recomendadas)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[P1] Incorporar tipo de cambio e IPC como controles macro
     → Verificar si la señal del 0km sobrevive
     → Series disponibles: BCRA, INDEC

[P2] Modelo de Corrección de Errores (ECM)
     → Cuantificar velocidad de ajuste al equilibrio de largo plazo
     → Requiere cointegración confirmada (ya disponible)

[P3] Recalibrar ISA con período base post-2024
     → Definir nuevo régimen en metricas.py
     → Comparar series bajo ambos regímenes

[P4] Dummy estacional diciembre
     → Controlar anomalía sistemática de fin de año
     → Limpiar señal del efecto cascada de ruido estacional
"""

print(hallazgos)


╔══════════════════════════════════════════════════════════════╗
║         MERCADO DE LIMONES ARGENTINO — ANÁLISIS CAUSAL      ║
║              Notebook 03 — Síntesis de Hallazgos            ║
╚══════════════════════════════════════════════════════════════╝

PERÍODO ANALIZADO: Enero 2023 — Mayo 2026 (41 meses)
RÉGIMEN BASE ISA:  2023_base (pre-shock Milei)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
HALLAZGOS CONFIRMADOS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[H1] COINTEGRACIÓN DETECTADA
     Las series ISA y Volumen 0km son individualmente no
     estacionarias I(1) pero comparten una relación estructural
     de largo plazo (Test Engle-Granger, p=0.0025).
     → Trabajar en niveles es válido. No hay regresión espuria.
     → Existe un "vínculo invisible" entre ambos mercados que
       los impide alejarse indefinidamente.

[H2] PRECEDENCIA TEMPORAL CONFIRMADA (Granger)
     El historial del volumen 0km contiene información útil
     para pred